# 03 · Calidad y preparación de datos
**Proyecto Integrador – Data Mining Tools (CC209)** · TP1 · **Paso 5**

Para cada problema: **detectar → decidir → justificar → registrar**.

**Principio para evitar fuga de información:** aquí solo se aplican reglas **fijas, fila a fila** (que no usan estadísticas calculadas con todo el dataset).
Todo lo que requiere aprender de los datos (imputar con la mediana, escalar, codificar, `log`) se hace después, dentro del `Pipeline`, ajustado solo con *train* (Paso 7).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_FILE = ROOT / "data" / "raw" / "kc_house_data.csv"
INTERIM_FILE = ROOT / "data" / "interim" / "kc_house_clean.csv"

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

df = pd.read_csv(RAW_FILE, dtype={"zipcode": "string"})
df["fecha"] = pd.to_datetime(df["date"], format="%Y%m%dT%H%M%S")
n_inicial = len(df)
print(f"Filas: {n_inicial:,} | Columnas: {df.shape[1]}")

Filas: 21,613 | Columnas: 22


## 1. Valores faltantes

In [2]:
nulos = df.isna().sum()
print("Columnas con nulos:", nulos[nulos > 0].to_dict() or "ninguna")

Columnas con nulos: ninguna


**Qué muestran los datos:** ninguna columna tiene nulos.

**Interpretación:** la ausencia de nulos **no** implica datos completos. En este dataset la información faltante está **codificada como 0** (se trata en las secciones 4 y 5). Por eso las decisiones de abajo tratan ciertos ceros como faltantes.

## 2. Duplicados

In [3]:
cols = [c for c in df.columns if c != "fecha"]
print("Filas duplicadas exactas:        ", df.duplicated(subset=cols).sum())
print("Mismo id y misma fecha:          ", df.duplicated(subset=["id", "fecha"]).sum())
print("id repetidos (ventas distintas): ", df["id"].duplicated().sum())

Filas duplicadas exactas:         0
Mismo id y misma fecha:           0
id repetidos (ventas distintas):  177


**Decisión:** no se elimina ninguna fila.
**Justificación:** no hay duplicados exactos. Los 177 `id` repetidos son **ventas distintas** de una misma vivienda (fechas distintas, precios distintos), y eliminarlas perdería información. El riesgo que generan es la fuga entre *train* y *test*, que se resuelve agrupando por `id` en el split (Paso 6).

## 3. Tipos de datos

In [4]:
tipos = pd.DataFrame({
    "tipo actual": df.dtypes.astype(str),
    "valores distintos": df.nunique(),
})
tipos

,tipo actual,valores distintos
id,int64,21436
date,str,372
price,float64,4028
bedrooms,int64,13
bathrooms,float64,30
sqft_living,int64,1038
sqft_lot,int64,9782
floors,float64,6
waterfront,int64,2
view,int64,5


**Decisiones**
| Variable | Problema | Decisión |
|---|---|---|
| `date` | texto | Convertida a fecha (`fecha`) |
| `zipcode` | número, pero es una **categoría** | Se trata como texto/categoría. Se codifica en el pipeline |
| `waterfront` | 0/1 | Se deja como indicador numérico |
| `view`, `condition`, `grade` | enteros ordinales | Se dejan numéricos: el orden importa y los árboles lo aprovechan |
| `id` | identificador | **No es predictor**. Se conserva solo para agrupar en el split |

### 3.1 Consistencia de las categorías
Se verifica que las variables categóricas y ordinales no tengan formatos mezclados, valores fuera de rango ni niveles duplicados con distinta escritura.

In [5]:
zip_ = df["zipcode"]
conteo_zip = zip_.value_counts()

consistencia = pd.Series({
    "zipcode con exactamente 5 caracteres": bool((zip_.str.len() == 5).all()),
    "zipcode que empieza con 98": bool(zip_.str.startswith("98").all()),
    "zipcode: niveles distintos": int(zip_.nunique()),
    "zipcode: ventas del nivel menos frecuente": int(conteo_zip.min()),
    "waterfront": sorted(df["waterfront"].unique().tolist()),
    "view": sorted(df["view"].unique().tolist()),
    "condition": sorted(df["condition"].unique().tolist()),
    "grade (niveles presentes)": sorted(df["grade"].unique().tolist()),
    "floors": sorted(df["floors"].unique().tolist()),
    "bathrooms: todos múltiplos de 0.25": bool((df["bathrooms"] % 0.25 == 0).all()),
    "lat y long dentro del condado": bool(df["lat"].between(47.0, 47.9).all() and df["long"].between(-122.6, -121.0).all()),
}, name="resultado")
consistencia.to_frame()

,resultado
zipcode con exactamente 5 caracteres,True
zipcode que empieza con 98,True
zipcode: niveles distintos,70
zipcode: ventas del nivel menos frecuente,50
waterfront,"[0, 1]"
view,"[0, 1, 2, 3, 4]"
condition,"[1, 2, 3, 4, 5]"
grade (niveles presentes),"[1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]"
floors,"[1.0, 1.5, 2.0, 2.5, 3.0, 3.5]"
bathrooms: todos múltiplos de 0.25,True


**Qué muestran los datos:** no hay categorías inconsistentes. Todos los `zipcode` tienen 5 dígitos y comienzan con 98 (70 niveles, el menos frecuente con 50 ventas); `waterfront` solo toma 0 y 1; `view` va de 0 a 4 y `condition` de 1 a 5; `grade` tiene 12 niveles distintos entre 1 y 13 (no aparece el 2); `floors` y `bathrooms` solo toman valores fraccionarios válidos (medios pisos, cuartos de baño); y las coordenadas caen dentro del condado.

**Decisión:** no se corrige ni se agrupa ninguna categoría. Todo nivel de `zipcode` tiene al menos 50 ventas, por lo que no hace falta fusionar niveles raros.

**Interpretación:** al ser todos códigos numéricos de un registro administrativo, la inconsistencia por escritura (por ejemplo, "Seattle" y "seattle") no aplica. Los problemas de calidad de este dataset son los ceros que codifican faltantes (sección 5), no las categorías.

## 4. Reglas lógicas y de negocio

In [6]:
anio_venta = df["fecha"].dt.year

reglas = pd.Series({
    "sqft_above + sqft_basement != sqft_living": int(((df["sqft_above"] + df["sqft_basement"]) != df["sqft_living"]).sum()),
    "sqft_basement > sqft_living": int((df["sqft_basement"] > df["sqft_living"]).sum()),
    "yr_renovated > 0 y < yr_built": int(((df["yr_renovated"] > 0) & (df["yr_renovated"] < df["yr_built"])).sum()),
    "yr_renovated posterior a 2015": int((df["yr_renovated"] > 2015).sum()),
    "yr_built posterior al año de venta": int((df["yr_built"] > anio_venta).sum()),
    "condition fuera de 1-5": int((~df["condition"].between(1, 5)).sum()),
    "grade fuera de 1-13": int((~df["grade"].between(1, 13)).sum()),
    "price <= 0": int((df["price"] <= 0).sum()),
    "bedrooms == 0": int((df["bedrooms"] == 0).sum()),
    "bathrooms == 0": int((df["bathrooms"] == 0).sum()),
}, name="casos")
reglas.to_frame()

,casos
sqft_above + sqft_basement != sqft_living,0
sqft_basement > sqft_living,0
yr_renovated > 0 y < yr_built,0
yr_renovated posterior a 2015,0
yr_built posterior al año de venta,12
condition fuera de 1-5,0
grade fuera de 1-13,0
price <= 0,0
bedrooms == 0,13
bathrooms == 0,10


**Qué muestran los datos**
- Las reglas de coherencia entre superficies, fechas y rangos se cumplen, **salvo** dos grupos: ventas de viviendas con `yr_built` posterior al año de venta (12 casos) y los ceros en dormitorios/baños (sección 5).

**Decisión sobre `yr_built` posterior a la venta (12 casos)**
- Se **conservan**. Es coherente con la preventa de obra nueva.
- La antigüedad se calcula como `max(0, año de venta − yr_built)` para no generar valores negativos.

## 5. Dormitorios y baños en cero, y el caso de 33 dormitorios
Se separan las viviendas con 0 dormitorios o 0 baños según su superficie.

In [7]:
cero = df[(df["bedrooms"] == 0) | (df["bathrooms"] == 0)]
grande = cero["sqft_living"] >= 1000
print(f"Con algún cero: {len(cero)} | de 1 000 pies² o más: {grande.sum()} | de menos de 1 000 pies²: {(~grande).sum()}")
cero[["price", "bedrooms", "bathrooms", "sqft_living", "grade"]].sort_values("sqft_living")

Con algún cero: 16 | de 1 000 pies² o más: 9 | de menos de 1 000 pies²: 7


,price,bedrooms,bathrooms,sqft_living,grade
19452,142000.0,0,0.00,290,1
18379,265000.0,0,0.75,384,4
4868,228000.0,0,1.00,390,4
5832,280000.0,1,0.00,600,3
1149,75000.0,1,0.00,670,3
10481,484000.0,1,0.00,690,7
14423,139950.0,0,0.00,844,7
3467,288000.0,0,1.50,1430,7
9854,235000.0,0,0.00,1470,7
3119,380000.0,0,0.00,1470,8


In [8]:
pies_por_dormitorio = (df["sqft_living"] / df["bedrooms"]).where(df["bedrooms"] > 0)
print("Pies² por dormitorio (menor valor):")
display(pies_por_dormitorio.nsmallest(3).to_frame("pies2_por_dormitorio").join(df[["bedrooms", "sqft_living", "bathrooms"]]))
print("Pies² por dormitorio, percentil 1 / mediana:", pies_por_dormitorio.quantile([.01, .5]).round(0).tolist())

Pies² por dormitorio (menor valor):


,pies2_por_dormitorio,bedrooms,sqft_living,bathrooms
15870,49.090909,33,1620,1.75
4203,163.333333,3,490,0.75
319,200.000000,4,800,1.00


Pies² por dormitorio, percentil 1 / mediana: [300.0, 577.0]


**Qué muestran los datos**
- 16 viviendas tienen 0 dormitorios o 0 baños. **9** miden entre 1 430 y 4 810 pies²; las otras **7** miden menos de 1 000 pies².
- La vivienda de **33 dormitorios** tiene solo 49 pies² por dormitorio. La mediana del dataset es 577, el percentil 1 es 300 y la siguiente vivienda más baja tiene 163 (3 dormitorios en 490 pies²), que sigue siendo posible en una unidad pequeña.

**Decisión**
| Caso | Regla | Tratamiento |
|---|---|---|
| 0 dormitorios o 0 baños en vivienda de ≥ 1 000 pies² | Una casa de ese tamaño no puede carecer de dormitorios o baños | Convertir el 0 a **faltante** (`NaN`); se imputa en el pipeline con estadísticas de *train* |
| 0 dormitorios o 0 baños en vivienda de < 1 000 pies² | Puede ser un estudio o una unidad pequeña | **Se conserva** el 0 |
| Menos de 100 pies² por dormitorio (solo el caso de 33; el siguiente tiene 163) | Físicamente implausible | Convertir `bedrooms` a **faltante** |
| 8 a 11 dormitorios con superficies grandes | Coherentes con casas grandes o multifamiliares | **Se conservan** |

**Justificación y riesgo**
- No se "corrige" el 33 a 3: sería adivinar. Tratarlo como faltante evita inventar el dato y delega la imputación en el pipeline, sin fuga.
- Se agrega un indicador `dato_dudoso` para poder revisar después si esos casos concentran los errores del modelo.
- **Incertidumbre:** el umbral de 1 000 pies² es un criterio razonable pero no verificable con estos datos. Los 7 casos pequeños podrían también ser errores; se conservan porque descartarlos sería igual de arbitrario.

In [9]:
df["dato_dudoso"] = False

gran_cero_dorm = (df["bedrooms"] == 0) & (df["sqft_living"] >= 1000)
gran_cero_bano = (df["bathrooms"] == 0) & (df["sqft_living"] >= 1000)
dormitorio_imposible = (df["sqft_living"] / df["bedrooms"].replace(0, np.nan)) < 100

df["bedrooms"] = df["bedrooms"].astype("float64")
df.loc[gran_cero_dorm | dormitorio_imposible, "bedrooms"] = np.nan
df.loc[gran_cero_bano, "bathrooms"] = np.nan
df.loc[gran_cero_dorm | dormitorio_imposible | gran_cero_bano, "dato_dudoso"] = True

print("bedrooms -> NaN:", int(df["bedrooms"].isna().sum()),
      "| bathrooms -> NaN:", int(df["bathrooms"].isna().sum()),
      "| filas marcadas dato_dudoso:", int(df["dato_dudoso"].sum()))
print("Máximo de dormitorios tras la regla:", df["bedrooms"].max())

bedrooms -> NaN: 10 | bathrooms -> NaN: 5 | filas marcadas dato_dudoso: 10
Máximo de dormitorios tras la regla: 11.0


## 6. Outliers y valores extremos
Se mide cuántos puntos quedan fuera de 1.5×IQR en escala original y en escala logarítmica.

In [10]:
def fuera_iqr(s):
    q1, q3 = s.quantile([.25, .75])
    i = q3 - q1
    return int(((s < q1 - 1.5 * i) | (s > q3 + 1.5 * i)).sum())

filas = []
for c in ["price", "sqft_living", "sqft_lot"]:
    filas.append((c, fuera_iqr(df[c]), fuera_iqr(np.log(df[c]))))
pd.DataFrame(filas, columns=["variable", "fuera de 1.5×IQR (original)", "fuera de 1.5×IQR (log)"]).set_index("variable")

,fuera de 1.5×IQR (original),fuera de 1.5×IQR (log)
variable,,
price,1146,333
sqft_living,572,110
sqft_lot,2425,2555


In [11]:
print("Mayores terrenos:")
display(df.nlargest(3, "sqft_lot")[["price", "sqft_lot", "sqft_living", "zipcode"]])
print("Ventas menores a 100 mil:", int((df["price"] < 100_000).sum()))
display(df.nsmallest(3, "price")[["price", "sqft_living", "grade", "zipcode"]])

Mayores terrenos:


,price,sqft_lot,sqft_living,zipcode
1719,700000.0,1651359,1300,98022
17319,190000.0,1164794,710,98014
7647,542500.0,1074218,3010,98027


Ventas menores a 100 mil: 25


,price,sqft_living,grade,zipcode
1149,75000.0,670,3,98022
15293,78000.0,780,5,98168
465,80000.0,430,4,98014


**Qué muestran los datos**
- Con 1.5×IQR en escala original quedan fuera **1 146** precios, **572** superficies habitables y **2 425** terrenos. Tras `log`, esos números bajan a **333** y **110** para precio y superficie; para `sqft_lot` no baja (2 555).
- Los casos más extremos son coherentes: un terreno de 1.65 millones de pies² con una casa de 1 300 pies² (zona rural); las ventas bajo 100 mil (25) son de viviendas pequeñas y de grade bajo.

**Decisión: no se elimina ningún outlier.**

**Justificación**
- La mayoría de los "outliers" en escala original son consecuencia de la asimetría, no de errores: se reducen con `log`.
- Eliminar los precios altos le quitaría al modelo justo los casos que interesa detectar como anomalías y sesgaría el error hacia las casas baratas.
- Los únicos valores tratados son los **imposibles** (sección 5). Los extremos legítimos se manejan con `log` y con modelos robustos.

**Riesgo:** los modelos lineales son sensibles a puntos influyentes (terrenos enormes, casas muy grandes). Se mitiga con `log1p`, y se revisará el efecto en el Paso 10.

## 7. Variables derivadas (reglas fijas fila a fila)
No usan estadísticas del dataset ni el precio, por lo que **no generan fuga**.

In [12]:
df["anio_venta"] = df["fecha"].dt.year
df["mes_venta"] = df["fecha"].dt.month
df["antiguedad"] = (df["anio_venta"] - df["yr_built"]).clip(lower=0)
df["renovada"] = (df["yr_renovated"] > 0).astype(int)
df["tiene_sotano"] = (df["sqft_basement"] > 0).astype(int)

df[["antiguedad", "renovada", "tiene_sotano"]].describe().T[["min", "50%", "max", "mean"]].round(2)

,min,50%,max,mean
antiguedad,0.0,40.0,115.0,43.32
renovada,0.0,0.0,1.0,0.04
tiene_sotano,0.0,0.0,1.0,0.39


**Decisión:** se crean `antiguedad`, `renovada` y `tiene_sotano`. Resuelven el hallazgo 7 del EDA (ceros que significan "no aplica").

**No se crea** `precio_por_pie2` ni ninguna variable que use `price`: contendría el objetivo y produciría fuga de información.

**Transformaciones que NO se hacen aquí** (se hacen en el pipeline, ajustadas solo con *train*): `log` del objetivo, `log1p` de superficies, imputación, escalado y codificación de `zipcode`.

## 8. Registro de decisiones

| # | Problema | Evidencia | Decisión | Justificación | Riesgo |
|---|---|---|---|---|---|
| 1 | Nulos | 0 nulos | Ninguna acción | El faltante está codificado como 0 | Falsa sensación de completitud |
| 2 | Duplicados | 0 exactos; 177 `id` repetidos | No eliminar; agrupar por `id` en el split | Son ventas distintas | Fuga *train*/*test* si no se agrupa |
| 3 | `zipcode` numérico | 70 valores | Tratar como categoría | Es un identificador, no una cantidad | 70 niveles: codificación a decidir (Paso 7) |
| 4 | `yr_built` > año de venta | 12 casos | Conservar; antigüedad mínima 0 | Coherente con preventa | Menor |
| 5 | 0 dormitorios/baños en ≥ 1 000 pies² | 9 viviendas | Convertir a `NaN` + imputar en pipeline | Físicamente imposible | Umbral arbitrario |
| 6 | 0 dormitorios/baños en < 1 000 pies² | 7 viviendas | Conservar | Puede ser estudio | Podrían ser errores |
| 7 | 33 dormitorios | 49 pies²/dormitorio (el siguiente: 163) | `bedrooms` a `NaN` | Implausible, no se adivina el valor real | Menor |
| 7b | Resultado de las reglas 5 y 7 | `bedrooms`: 10 `NaN`; `bathrooms`: 5 `NaN`; 10 filas con `dato_dudoso` | Imputar en el pipeline con estadísticas de *train* | Evita fuga de información | Impacto mínimo: 10 de 21 613 filas (0.05 %) |
| 8 | Outliers de precio y superficie | 1 146 / 572 (original); 333 / 110 (log) | No eliminar | Efecto de la asimetría; son el objeto de estudio | Sensibilidad de modelos lineales |
| 9 | Ceros "no aplica" | `yr_renovated`, `sqft_basement` | Crear `renovada`, `tiene_sotano` | Separa "no aplica" de una cantidad | Ninguno |

## 9. Guardado del dataset limpio
Se guarda en `data/interim/` (no se versiona). El crudo de `data/raw/` permanece intacto.

In [13]:
orden = ["id", "fecha", "price", "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "waterfront",
         "view", "condition", "grade", "sqft_above", "sqft_basement", "yr_built", "yr_renovated",
         "zipcode", "lat", "long", "sqft_living15", "sqft_lot15",
         "anio_venta", "mes_venta", "antiguedad", "renovada", "tiene_sotano", "dato_dudoso"]
limpio = df[orden]

assert len(limpio) == n_inicial, "No se debe perder ninguna fila"
assert limpio["price"].notna().all()

limpio.to_csv(INTERIM_FILE, index=False)
print(f"Guardado: {INTERIM_FILE.relative_to(ROOT)} | {limpio.shape[0]:,} filas x {limpio.shape[1]} columnas")
print("Nulos por columna (solo las que tienen):", limpio.isna().sum()[lambda s: s > 0].to_dict())

Guardado: data\interim\kc_house_clean.csv | 21,613 filas x 27 columnas
Nulos por columna (solo las que tienen): {'bedrooms': 10, 'bathrooms': 5}
